<a href="https://colab.research.google.com/github/dustu15/AI-build-Training/blob/trainingday/day3_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
imdb = load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
small_train_dataset = imdb["train"].shuffle(seed=42).select([i for i in list(range(3000))])
small_test_dataset = imdb["test"].shuffle(seed=42).select([i for i in list(range(300))])

In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
   return tokenizer(examples["text"], truncation=True)

tokenized_train = small_train_dataset.map(preprocess_function, batched=True)
tokenized_test = small_test_dataset.map(preprocess_function, batched=True)

In [4]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [5]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
!pip install evaluate

In [7]:
import numpy as np
from evaluate import load

def compute_metrics(eval_pred):
    accuracy_metric = load("accuracy")
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [8]:
from huggingface_hub import notebook_login
notebook_login()

In [9]:
from transformers import TrainingArguments, Trainer

In [10]:
repo_name = "my_model"

training_args = TrainingArguments(
   output_dir=repo_name,
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=2,
   weight_decay=0.01,
   save_strategy="epoch",
   push_to_hub=True,
)

In [12]:
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [13]:
trainer.train()

trainer.evaluate()

trainer.push_to_hub()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...y_model/training_args.bin: 100%|##########| 5.14kB / 5.14kB            

  ...y_model/model.safetensors:  13%|#2        | 33.6MB /  268MB            

CommitInfo(commit_url='https://huggingface.co/dustu15/my_model/commit/a3a5de4d65affbb749b2733684910eb795aff34d', commit_message='End of training', commit_description='', oid='a3a5de4d65affbb749b2733684910eb795aff34d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dustu15/my_model', endpoint='https://huggingface.co', repo_type='model', repo_id='dustu15/my_model'), pr_revision=None, pr_num=None)

In [14]:
from transformers import pipeline
model=pipeline("sentiment-analysis",model="dustu15/my_model")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
model(["I love this color and its so warming","This movie is boring","This movie is great"])

[{'label': 'LABEL_1', 'score': 0.9608505964279175},
 {'label': 'LABEL_0', 'score': 0.978640615940094},
 {'label': 'LABEL_1', 'score': 0.9609843492507935}]

# Task
Load the Excel file located at "/content/BankReviews.xlsx", apply the sentiment analysis model to the 'review' column of the loaded data to predict the sentiment, add a new column named 'prediction' to the data containing the sentiment predictions, and then display the updated data.

## Load Excel File

### Subtask:
Load the data from '/content/BankReviews.xlsx' into a pandas DataFrame.


**Reasoning**:
To load the Excel file, I will import the pandas library and use its `read_excel` function. Then, I will display the first few rows to confirm the data is loaded correctly.



In [17]:
import pandas as pd
df = pd.read_excel('/content/BankReviews.xlsx')
df.head()

,Date,Stars,Reviews,BankName
0,2017-04-10,5,"Great job, Wyndham Capital! Each person was pr...",Wyndham Capital Mortgage
1,2017-02-10,5,Matthew Richardson is professional and helpful...,Wyndham Capital Mortgage
2,2017-08-21,5,We had a past experience with Wyndham Mortgage...,Wyndham Capital Mortgage
3,2017-12-17,5,We have been dealing with Brad Thomka from the...,Wyndham Capital Mortgage
4,2016-05-27,5,I can't express how grateful I am for the supp...,Wyndham Capital Mortgage


## Apply Sentiment Analysis Model

### Subtask:
Apply the existing sentiment analysis pipeline (`model` from cell `2U_8SeFJfRMZ`) to the 'review' column of the DataFrame to get predictions.


**Reasoning**:
First, I will extract the 'Reviews' column from the DataFrame and convert it to a list of strings. Then, I will apply the `model` pipeline to this list of reviews to obtain the sentiment predictions and print the first 5 predictions to inspect their structure.



In [18]:
reviews = df['Reviews'].tolist()
predictions = model(reviews)
print(predictions[0:5])

Token indices sequence length is longer than the specified maximum sequence length for this model (693 > 512). Running this sequence through the model will result in indexing errors


RuntimeError: The size of tensor a (693) must match the size of tensor b (512) at non-singleton dimension 1

**Reasoning**:
The previous code failed because some review texts were longer than the model's maximum sequence length (512), causing a `RuntimeError`. To fix this, I need to explicitly tell the pipeline to truncate longer sequences by setting `truncation=True` when calling the model.



In [19]:
reviews = df['Reviews'].tolist()
predictions = model(reviews, truncation=True)
print(predictions[0:5])

[{'label': 'LABEL_1', 'score': 0.9688965082168579}, {'label': 'LABEL_1', 'score': 0.9548594355583191}, {'label': 'LABEL_1', 'score': 0.9192332029342651}, {'label': 'LABEL_1', 'score': 0.9482355117797852}, {'label': 'LABEL_1', 'score': 0.9707951545715332}]


**Reasoning**:
Now that the sentiment predictions have been generated and their structure inspected, the next step is to extract the 'label' from each prediction and add it as a new column named 'prediction' to the original DataFrame. Finally, I will display the updated DataFrame to verify the new column.



In [20]:
df['prediction'] = [p['label'] for p in predictions]
df.head()

,Date,Stars,Reviews,BankName,prediction
0,2017-04-10,5,"Great job, Wyndham Capital! Each person was pr...",Wyndham Capital Mortgage,LABEL_1
1,2017-02-10,5,Matthew Richardson is professional and helpful...,Wyndham Capital Mortgage,LABEL_1
2,2017-08-21,5,We had a past experience with Wyndham Mortgage...,Wyndham Capital Mortgage,LABEL_1
3,2017-12-17,5,We have been dealing with Brad Thomka from the...,Wyndham Capital Mortgage,LABEL_1
4,2016-05-27,5,I can't express how grateful I am for the supp...,Wyndham Capital Mortgage,LABEL_1


## Final Task

### Subtask:
Confirm that the sentiment predictions have been successfully added to the Excel data.


## Summary:

### Q&A
Yes, the sentiment predictions have been successfully added to the Excel data. A new column named 'prediction' was added to the DataFrame, containing the sentiment labels for each review.

### Data Analysis Key Findings
*   The Excel file `BankReviews.xlsx` was successfully loaded into a pandas DataFrame.
*   An initial `RuntimeError` occurred when applying the sentiment analysis model due to some review texts exceeding the model's maximum supported sequence length of 512 tokens.
*   This error was resolved by enabling `truncation=True` during the sentiment analysis prediction, allowing the model to process all reviews.
*   The sentiment predictions, specifically the 'label' from the model's output (e.g., 'LABEL\_1'), were successfully extracted and added as a new column named 'prediction' to the DataFrame.

### Insights or Next Steps
*   When working with NLP models, it is crucial to anticipate and handle input constraints like maximum sequence length to ensure robust processing.
*   The updated DataFrame, now including sentiment predictions, can be used for further analysis such as understanding customer sentiment distribution, identifying common themes within positive or negative reviews, or segmenting customers based on their expressed sentiment.


In [21]:
df.sample(10)


,Date,Stars,Reviews,BankName,prediction
173,2017-04-24,5,_x000D_\nWe received offers from multiple bank...,North American Savings Bank,LABEL_1
274,2017-05-02,5,_x000D_\nChris Waymire and his team were spect...,Triumph Lending,LABEL_1
490,2016-05-12,5,_x000D_\nOur refinance process was a dream onc...,North American Savings Bank,LABEL_1
72,2016-04-13,5,_x000D_\nIt was fast easy and did not have to ...,Reliance First Capital,LABEL_1
305,2016-10-22,5,_x000D_\nI dealt with Robert McClung and when ...,LoanSnap,LABEL_1
76,2017-10-19,5,_x000D_\nThank you for all of your help Kelly!...,Reliance First Capital,LABEL_1
476,2016-10-29,5,_x000D_\nI was hesitant to use a non-local mor...,North American Savings Bank,LABEL_0
140,2017-10-11,5,_x000D_\nWorked with Jason & June & it couldn’...,Pacific Beneficial Mortgage Co,LABEL_1
470,2016-12-05,5,_x000D_\nExtremely smooth process with no surp...,North American Savings Bank,LABEL_1
499,2016-03-22,1,_x000D_\nThis Lender contacted my previous pho...,North American Savings Bank,LABEL_0


### Save the Updated DataFrame to an Excel File

In [22]:
output_filename = 'BankReviews_with_predictions.xlsx'
df.to_excel(output_filename, index=False)
print(f"DataFrame saved to {output_filename}")

DataFrame saved to BankReviews_with_predictions.xlsx
